# Uncertainty expansion — direct import test

This notebook imports the expansion machinery straight from this repository
and solves two models end to end:

1. the **Chapter 11 AK economy** (chapter equations, appendix parameters);
2. the **internal-habit model** from the book's *Uncertainty Expansion —
   Computation Process* appendix.

Each model is declared **in a cell below** through the user-facing `Model`
layer; underneath, the original `uncertain_expansion` engine does the
solving. **Runtime → Run all** (~1 min).

In [ ]:
%pip -q install autograd
import os, sys
if not os.path.isdir("QuantMFR-Colab"):
    !git clone -q --depth 1 https://github.com/as7391746/QuantMFR-Colab
sys.path.insert(0, os.path.abspath("QuantMFR-Colab/src"))        # the engine, same src/ layout as upstream
sys.path.insert(0, os.path.abspath("QuantMFR-Colab/expansion"))  # the declaration layer on top

import numpy as np
import sympy as sp

from uncertain_expansion import uncertain_expansion   # direct import, exactly as in the book appendix
from expansion import Model                            # the user-facing layer

import uncertain_expansion as _engine
print("engine loaded from:", _engine.__file__)

## 1. AK economy (Chapter 11)

States $(Z_1, Z_2)$, controls $(D_1, D_2)$, three shocks; equations exactly
as the chapter prints them (shock loadings without the noise scale,
Itô-correction terms with an explicit $q^2$). Parameters are the
quarterly values of the chapter appendix.

In [ ]:
SQRT3 = np.sqrt(3.0)
PARAMS = {
    "beta": np.exp(-0.0025), "gamma": 8.001, "rho": 1.001,
    "alpha": 0.02305, "zeta": 32.0, "iota_k": 0.01, "nu_k": 0.01,
    "nu_1": 0.014, "nu_2": 0.0485, "mu_2": 6.3e-6,
    "sigma_k": SQRT3 * np.array([0.92, 0.40, 0.0]),
    "sigma_1": SQRT3 * np.array([0.0, 5.70, 0.0]),
    "sigma_2": SQRT3 * np.array([0.0, 0.0, 0.00031]),
}

m = Model(states=["Z1", "Z2"], controls=["D1", "D2"], shocks=3)
Z1, Z2, D1, D2 = m["Z1"], m["Z2"], m["D1"], m["D2"]
p, q = m.p, m.q

m.state["Z1"] = Z1 - p.nu_1 * Z1 + sp.exp(Z2 / 2) * m.dot("sigma_1")
m.state["Z2"] = (Z2 - p.nu_2 * (1 - p.mu_2 * sp.exp(-Z2))
                 - q**2 / 2 * m.norm2("sigma_2") * sp.exp(-Z2)
                 + sp.exp(-Z2 / 2) * m.dot("sigma_2"))
m.growth = ((1 / p.zeta) * sp.log(1 + p.zeta * D2) + p.nu_k * Z1 - p.iota_k
            - q**2 / 2 * m.norm2("sigma_k") * sp.exp(Z2)
            + sp.exp(Z2 / 2) * m.dot("sigma_k"))
m.consumption = sp.log(D1)
m.constraint = p.alpha - D1 - D2

sol = m.solve(PARAMS, start={"D1": 0.004, "D2": 0.019,
                             "Z2": float(np.log(6.3e-6)), "growth": 0.005})
curve = sol.price_elasticity(shock=1, T=160, quantile=0.5)
print("price elasticity (growth shock, median state):")
print(f"  t = 1 quarter : {curve[0]:.8f}   (reference 0.11901239)")
print(f"  t = 40 years  : {curve[-1]:.8f}   (reference 0.13483263)")
assert abs(curve[0] - 0.11901239) < 1e-6 and abs(curve[-1] - 0.13483263) < 1e-6
print("AK check: PASS")

## 2. Internal-habit model (the book's appendix example)

Adds a habit state $X = \log(H/K)$ and replaces log consumption with a CES
aggregate of the consumption input and the habit stock. Equations and
parameters transcribed from the appendix notebook (time step
$\epsilon = 0.25$, annual drifts, $\sqrt{12}$ volatilities).

In [ ]:
HPARAMS = {
    "epsilon": 0.25, "beta": float(np.exp(-0.01 * 0.25)),
    "gamma": 1.001, "rho": 1.001, "a": 0.0922,
    "phi_1": 1.0 / 8.0, "phi_2": 8.0,
    "alpha_k": 0.04, "beta_k": 0.04, "beta_z": 0.056, "beta_2": 0.194,
    "mu_2": 6.3e-6,
    "sigma_k": np.sqrt(12.0) * np.array([0.92, 0.40, 0.0]),
    "sigma_z": np.sqrt(12.0) * np.array([0.0, 5.70, 0.0]),
    "sigma_y": np.sqrt(12.0) * np.array([0.0, 0.0, 0.00031]),
    "nu_h": 0.1 * 0.25, "tau": 1.01, "llambda": -0.0,
}

mh = Model(states=["Z", "Y", "X"], controls=["imh", "imk"], shocks=3)
Z, Y, X = mh["Z"], mh["Y"], mh["X"]
imh, imk = mh["imh"], mh["imk"]
p, q = mh.p, mh.q
seps = sp.sqrt(p.epsilon)

mh.state["Z"] = Z - p.epsilon * p.beta_z * Z + seps * sp.exp(Y / 2) * mh.dot("sigma_z")
mh.state["Y"] = (Y - p.epsilon * p.beta_2 * (1 - p.mu_2 * sp.exp(-Y))
                 - q**2 / 2 * mh.norm2("sigma_y") * sp.exp(-Y) * p.epsilon
                 + seps * sp.exp(-Y / 2) * mh.dot("sigma_y"))
capital = (p.epsilon * (p.phi_1 * sp.log(1 + p.phi_2 * imk)
                        - p.alpha_k + p.beta_k * Z
                        - q**2 / 2 * mh.norm2("sigma_k") * sp.exp(Y))
           + seps * sp.exp(Y / 2) * mh.dot("sigma_k"))
mh.growth = capital
mh.state["X"] = (sp.log(sp.exp(-p.nu_h + X) + (1 - sp.exp(-p.nu_h)) * imh)
                 - capital)
mh.consumption = (1 / (1 - p.tau)
                  * sp.log((1 - p.llambda) * imh**(1 - p.tau)
                           + p.llambda * sp.exp((1 - p.tau) * X)))
mh.constraint = p.a - imh - imk

APPENDIX_SS = np.array([-4.12667104, 0.15503472, 0.01613651, 0.07606349,
                        0.60576715, 0.0, -0.0, 1.0, 0.00485334, -0.0,
                        -11.97496092, -4.30652983])
solh = mh.solve(HPARAMS,
                start={"imh": 0.016, "imk": 0.076,
                       "Y": float(np.log(6.3e-6)), "X": -4.3,
                       "growth": 0.005},
                tol=1e-5)
ss = np.asarray(solh.raw["ss"], float).flatten()
diff = np.max(np.abs(ss - APPENDIX_SS))
print(f"steady state vs the appendix notebook's stored output: "
      f"max abs diff {diff:.2e}")
assert diff < 1e-6
print("habit check: PASS")

Two different models, one declaration layer, one engine. A new model is a
new set of equations in a cell like the ones above; naming conventions,
argument ordering, and the steady-state starting vector are handled by the
layer (with hard errors — not silent corruption — when a name would collide
with the engine's internals).